# Week 2 Notebook 02 — pandapower 三相不平衡潮流建模与验证

**课程主题**：三相不平衡潮流、对称分量、VUF、逐相功率平衡、结果交叉验证。  
**科研故事线**：Microgrid static security prediction with phase-aware physics-informed AI。  
**本 Notebook 的目标**：用一个小型低压 microgrid-like feeder 学会 `pandapower.runpp_3ph()`，并用物理公式验证结果不是黑箱。

> Week 1 用 balanced `pp.runpp()` 熟悉 pandapower。Week 2 进入 `runpp_3ph()`、`asymmetric_load`、`asymmetric_sgen` 和三相结果表。Week 4 会把本周的 base-case 三相检查扩展成 N-1 violation label。

## 0. Learning outcomes

完成本 Notebook 后，你应该能够：

1. 构建一个最小三相 LV feeder；
2. 使用 `asymmetric_load` 表示 A/B/C 三相不同负荷；
3. 使用 `asymmetric_sgen` 表示相别不均匀 PV / DER；
4. 运行 `runpp_3ph()`；
5. 读取 `res_bus_3ph`, `res_line_3ph`, `res_ext_grid_3ph`；
6. 从 A/B/C 相量手算 positive / negative / zero sequence；
7. 复核 `unbalance_percent`；
8. 逐相验证有功/无功功率平衡；
9. 用 $S=VI^*$ 复核线路相电流；
10. 用线路额定电流复核 loading percent；
11. 用 balanced case 对 `runpp()` 和 `runpp_3ph()` 做交叉验证。

## 1. Setup

如果本地环境没有安装 pandapower，可以先运行：

```bash
pip install pandapower
```

本周只构建一个 0.4 kV 的小型 LV feeder，不引入变压器。这样做是为了把注意力集中在三件事上：

1. 三相不平衡负荷和 PV 如何写入 pandapower；
2. `runpp_3ph()` 输出如何读取；
3. 输出如何用基本物理进行 proof / cross-validation。

Week 3 再引入更接近真实微电网的数据生成流程。

In [ ]:
from __future__ import annotations

import copy
import math
import warnings
from pathlib import Path
from typing import Dict, Iterable, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pandapower as pp
except ImportError as exc:
    raise ImportError(
        "pandapower is required for this notebook. Install it with `pip install pandapower`."
    ) from exc

pd.set_option("display.max_columns", 120)
pd.set_option("display.precision", 6)

print(f"pandapower version: {pp.__version__}")

## 2. 教学阈值和数值容差

下面的安全阈值是**教学默认值**，用于 Week 2 的演示。正式论文中应说明阈值来源，或者按具体微电网运行规程设置。

数值容差用于 proof 和交叉验证。如果某个 proof 超过容差，代码会抛出 `AssertionError`，说明当前模型、单位换算或符号约定需要检查。

In [ ]:
PHASES = ["a", "b", "c"]

# Teaching security thresholds, not universal standards.
V_MIN_PU = 0.95
V_MAX_PU = 1.05
LOADING_MAX_PERCENT = 100.0
VUF_MAX_PERCENT = 2.0

# Numerical tolerances for proofs / cross-validation.
TOL_POWER_MW = 1e-5
TOL_POWER_MVAR = 1e-5
TOL_VUF_PERCENT = 1e-8
TOL_CURRENT_KA = 1e-9
TOL_LOADING_PERCENT = 1e-8
TOL_BALANCED_VM_PU = 1e-5
TOL_BALANCED_LOADING_PERCENT = 1e-3

## 3. 构建一个最小三相教学 feeder

本周的 feeder：

```text
PCC / Slack Bus  ── Line 0 ── Bus 1 ── Line 1 ── Bus 2
                                      │
                                      ├── asymmetric loads
                                      └── phase-specific PV / DER
```

注意：

- `vn_kv=0.4` 表示 low-voltage feeder 的 nominal line-to-line voltage；
- 三相潮流中 line 需要 zero-sequence 参数：`r0_ohm_per_km`, `x0_ohm_per_km`, `c0_nf_per_km`；
- `asymmetric_load` 中负荷为正；
- `asymmetric_sgen` 中正有功表示发电注入；
- 这里的线路参数是教学用的合理数量级，不代表某一种具体电缆型号。

In [ ]:
def create_three_phase_teaching_feeder(unbalanced: bool = True) -> pp.pandapowerNet:
    """Create a small LV three-phase feeder for Week 2.

    Parameters
    ----------
    unbalanced:
        If True, create deliberately asymmetric per-phase loads and PV.
        If False, create a perfectly balanced three-phase case for cross-validation.

    Returns
    -------
    pandapowerNet
        A pandapower network ready for pp.runpp_3ph().
    """
    net = pp.create_empty_network(name="week2_three_phase_teaching_feeder", sn_mva=1.0)

    # Buses. For LV systems, vn_kv is usually line-to-line nominal voltage.
    b_pcc = pp.create_bus(net, vn_kv=0.4, name="PCC_0p4kV")
    b_1 = pp.create_bus(net, vn_kv=0.4, name="LV_Bus_1")
    b_2 = pp.create_bus(net, vn_kv=0.4, name="LV_Bus_2")

    # External grid / PCC. Three-phase power flow requires short-circuit and sequence-related fields.
    pp.create_ext_grid(
        net,
        bus=b_pcc,
        vm_pu=1.0,
        va_degree=0.0,
        name="PCC_slack",
        s_sc_max_mva=10.0,
        s_sc_min_mva=8.0,
        rx_max=0.10,
        rx_min=0.10,
        x0x_max=1.0,
        x0x_min=1.0,
        r0x0_max=0.10,
        r0x0_min=0.10,
    )

    # Teaching line parameters. These are plausible teaching values, not a specific cable data sheet.
    line_specs = [
        (b_pcc, b_1, 0.08, "Line_0_PCC_to_Bus1"),
        (b_1, b_2, 0.06, "Line_1_Bus1_to_Bus2"),
    ]
    for from_bus, to_bus, length_km, name in line_specs:
        pp.create_line_from_parameters(
            net,
            from_bus=from_bus,
            to_bus=to_bus,
            length_km=length_km,
            r_ohm_per_km=0.642,
            x_ohm_per_km=0.083,
            c_nf_per_km=210.0,
            max_i_ka=0.20,
            r0_ohm_per_km=1.80,
            x0_ohm_per_km=0.25,
            c0_nf_per_km=70.0,
            name=name,
        )

    if unbalanced:
        # Bus 1: unbalanced residential / critical loads.
        pp.create_asymmetric_load(
            net,
            bus=b_1,
            p_a_mw=0.010,
            p_b_mw=0.008,
            p_c_mw=0.006,
            q_a_mvar=0.0020,
            q_b_mvar=0.0016,
            q_c_mvar=0.0012,
            type="wye",
            name="Bus1_unbalanced_load",
        )
        # Bus 2: heavier B-phase load.
        pp.create_asymmetric_load(
            net,
            bus=b_2,
            p_a_mw=0.006,
            p_b_mw=0.012,
            p_c_mw=0.004,
            q_a_mvar=0.0012,
            q_b_mvar=0.0024,
            q_c_mvar=0.0008,
            type="wye",
            name="Bus2_unbalanced_load",
        )
        # Phase-specific PV / DER at Bus 2.
        pp.create_asymmetric_sgen(
            net,
            bus=b_2,
            p_a_mw=0.003,
            p_b_mw=0.000,
            p_c_mw=0.002,
            q_a_mvar=0.0,
            q_b_mvar=0.0,
            q_c_mvar=0.0,
            type="wye",
            name="Bus2_phase_specific_PV",
        )
    else:
        # Perfectly balanced case used to compare runpp_3ph with runpp.
        pp.create_asymmetric_load(
            net,
            bus=b_1,
            p_a_mw=0.008,
            p_b_mw=0.008,
            p_c_mw=0.008,
            q_a_mvar=0.0016,
            q_b_mvar=0.0016,
            q_c_mvar=0.0016,
            type="wye",
            name="Bus1_balanced_load",
        )
        pp.create_asymmetric_load(
            net,
            bus=b_2,
            p_a_mw=0.006,
            p_b_mw=0.006,
            p_c_mw=0.006,
            q_a_mvar=0.0012,
            q_b_mvar=0.0012,
            q_c_mvar=0.0012,
            type="wye",
            name="Bus2_balanced_load",
        )
        pp.create_asymmetric_sgen(
            net,
            bus=b_2,
            p_a_mw=0.002,
            p_b_mw=0.002,
            p_c_mw=0.002,
            q_a_mvar=0.0,
            q_b_mvar=0.0,
            q_c_mvar=0.0,
            type="wye",
            name="Bus2_balanced_PV",
        )

    return net


net = create_three_phase_teaching_feeder(unbalanced=True)
print(net)

## 4. Proof 0 — 输入网络结构验证

在运行潮流前先检查输入结构，避免常见错误：

- element 的 bus index 不存在；
- line 缺少 zero-sequence 参数；
- 线路长度、额定电流非正；
- `asymmetric_load` / `asymmetric_sgen` 的相功率列缺失；
- external grid 缺少三相潮流常用参数。

In [ ]:
def _require_columns(df: pd.DataFrame, table_name: str, columns: Iterable[str]) -> None:
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise AssertionError(f"{table_name} is missing required columns: {missing}")


def validate_input_network(net: pp.pandapowerNet) -> None:
    """Validate the Week 2 input network before running runpp_3ph."""
    if len(net.ext_grid) != 1:
        raise AssertionError("This teaching feeder expects exactly one ext_grid / PCC.")

    if len(net.bus) < 2:
        raise AssertionError("Network should contain at least two buses.")

    # Check bus references.
    valid_buses = set(net.bus.index)
    for table_name in ["ext_grid", "asymmetric_load", "asymmetric_sgen"]:
        if hasattr(net, table_name) and len(net[table_name]) > 0:
            if "bus" not in net[table_name].columns:
                raise AssertionError(f"{table_name} must contain a bus column.")
            bad = set(net[table_name]["bus"]) - valid_buses
            if bad:
                raise AssertionError(f"{table_name} contains invalid bus references: {bad}")

    for col in ["from_bus", "to_bus"]:
        bad = set(net.line[col]) - valid_buses
        if bad:
            raise AssertionError(f"net.line.{col} contains invalid bus references: {bad}")

    # Check line parameters required by three-phase calculations.
    _require_columns(
        net.line,
        "net.line",
        [
            "r_ohm_per_km",
            "x_ohm_per_km",
            "c_nf_per_km",
            "r0_ohm_per_km",
            "x0_ohm_per_km",
            "c0_nf_per_km",
            "max_i_ka",
            "length_km",
        ],
    )
    if (net.line["length_km"] <= 0).any():
        raise AssertionError("All line lengths must be positive.")
    if (net.line["max_i_ka"] <= 0).any():
        raise AssertionError("All line current limits max_i_ka must be positive.")

    # Check three-phase load/generation columns.
    load_cols = [f"p_{ph}_mw" for ph in PHASES] + [f"q_{ph}_mvar" for ph in PHASES]
    if len(net.asymmetric_load) > 0:
        _require_columns(net.asymmetric_load, "net.asymmetric_load", load_cols)
    if len(net.asymmetric_sgen) > 0:
        _require_columns(net.asymmetric_sgen, "net.asymmetric_sgen", load_cols)

    # Check ext_grid parameters that runpp_3ph commonly needs.
    _require_columns(
        net.ext_grid,
        "net.ext_grid",
        ["s_sc_max_mva", "s_sc_min_mva", "rx_max", "rx_min", "x0x_max", "x0x_min", "r0x0_max", "r0x0_min"],
    )

    print("Input network validation passed.")


validate_input_network(net)

## 5. 运行三相潮流

`runpp_3ph()` 输出的关键表：

- `res_bus_3ph`: A/B/C 三相电压幅值、角度、各相 P/Q、`unbalance_percent`；
- `res_line_3ph`: A/B/C 三相线路功率、电流、loading、loss；
- `res_ext_grid_3ph`: PCC 各相注入功率；
- `res_asymmetric_load_3ph`: 各三相负荷的逐相功率；
- `res_asymmetric_sgen_3ph`: 各三相静态发电的逐相功率。

In [ ]:
def validate_result_tables(net: pp.pandapowerNet) -> None:
    """Validate that important runpp_3ph result tables and columns exist."""
    if not getattr(net, "converged", False):
        raise AssertionError("Power flow has not converged.")

    _require_columns(
        net.res_bus_3ph,
        "net.res_bus_3ph",
        [
            "vm_a_pu", "va_a_degree",
            "vm_b_pu", "va_b_degree",
            "vm_c_pu", "va_c_degree",
            "unbalance_percent",
        ],
    )
    _require_columns(
        net.res_line_3ph,
        "net.res_line_3ph",
        [
            "p_a_from_mw", "q_a_from_mvar", "p_b_from_mw", "q_b_from_mvar", "p_c_from_mw", "q_c_from_mvar",
            "p_a_to_mw", "q_a_to_mvar", "p_b_to_mw", "q_b_to_mvar", "p_c_to_mw", "q_c_to_mvar",
            "pl_a_mw", "ql_a_mvar", "pl_b_mw", "ql_b_mvar", "pl_c_mw", "ql_c_mvar",
            "i_a_from_ka", "i_b_from_ka", "i_c_from_ka",
            "i_a_to_ka", "i_b_to_ka", "i_c_to_ka",
            "i_a_ka", "i_b_ka", "i_c_ka",
            "loading_a_percent", "loading_b_percent", "loading_c_percent", "loading_percent",
        ],
    )
    _require_columns(
        net.res_ext_grid_3ph,
        "net.res_ext_grid_3ph",
        [f"p_{ph}_mw" for ph in PHASES] + [f"q_{ph}_mvar" for ph in PHASES],
    )
    if len(net.asymmetric_load) > 0:
        _require_columns(
            net.res_asymmetric_load_3ph,
            "net.res_asymmetric_load_3ph",
            [f"p_{ph}_mw" for ph in PHASES] + [f"q_{ph}_mvar" for ph in PHASES],
        )
    if len(net.asymmetric_sgen) > 0:
        _require_columns(
            net.res_asymmetric_sgen_3ph,
            "net.res_asymmetric_sgen_3ph",
            [f"p_{ph}_mw" for ph in PHASES] + [f"q_{ph}_mvar" for ph in PHASES],
        )

    print("Result table validation passed.")


def run_three_phase_power_flow(net: pp.pandapowerNet) -> pp.pandapowerNet:
    """Run three-phase power flow and validate results."""
    validate_input_network(net)
    pp.runpp_3ph(
        net,
        calculate_voltage_angles=True,
        init="auto",
        max_iteration=50,
        tolerance_mva=1e-8,
    )
    validate_result_tables(net)
    return net


net = run_three_phase_power_flow(net)

In [ ]:
print("res_bus_3ph")
display(net.res_bus_3ph)

print("res_line_3ph")
display(net.res_line_3ph)

print("res_ext_grid_3ph")
display(net.res_ext_grid_3ph)

## 6. Proof 1 — 从对称分量手算 voltage unbalance

对称分量变换：

$$
a=e^{j2\pi/3}
$$

$$
\begin{bmatrix}
V^{(0)} \\
V^{(1)} \\
V^{(2)}
\end{bmatrix}
=
\frac{1}{3}
\begin{bmatrix}
1 & 1 & 1 \\
1 & a & a^2 \\
1 & a^2 & a
\end{bmatrix}
\begin{bmatrix}
V_a \\
V_b \\
V_c
\end{bmatrix}
$$

VUF 定义：

$$
\mathrm{VUF}=100\frac{|V^{(2)}|}{|V^{(1)}|}
$$

我们将手算 VUF，并与 `net.res_bus_3ph["unbalance_percent"]` 比较。

In [ ]:
def compute_bus_phase_phasors(net: pp.pandapowerNet) -> pd.DataFrame:
    """Return complex phase voltages from res_bus_3ph in pu."""
    rb = net.res_bus_3ph
    out = pd.DataFrame(index=rb.index)
    for ph in PHASES:
        vm = rb[f"vm_{ph}_pu"].to_numpy(dtype=float)
        va = np.deg2rad(rb[f"va_{ph}_degree"].to_numpy(dtype=float))
        out[f"V_{ph}"] = vm * np.exp(1j * va)
    return out


def compute_symmetrical_components(net: pp.pandapowerNet) -> pd.DataFrame:
    """Compute zero-, positive-, and negative-sequence voltages from phase phasors."""
    ph = compute_bus_phase_phasors(net)
    alpha = np.exp(1j * 2 * np.pi / 3)

    Va = ph["V_a"].to_numpy()
    Vb = ph["V_b"].to_numpy()
    Vc = ph["V_c"].to_numpy()

    V0 = (Va + Vb + Vc) / 3
    V1 = (Va + alpha * Vb + alpha**2 * Vc) / 3
    V2 = (Va + alpha**2 * Vb + alpha * Vc) / 3

    seq = pd.DataFrame(index=net.bus.index)
    seq["V0_abs_pu"] = np.abs(V0)
    seq["V1_abs_pu"] = np.abs(V1)
    seq["V2_abs_pu"] = np.abs(V2)
    seq["VUF_calc_percent"] = 100 * seq["V2_abs_pu"] / seq["V1_abs_pu"]
    seq["VUF_pandapower_percent"] = net.res_bus_3ph["unbalance_percent"].to_numpy()
    seq["VUF_error_percent"] = seq["VUF_calc_percent"] - seq["VUF_pandapower_percent"]
    return seq


def validate_vuf_against_pandapower(net: pp.pandapowerNet, atol: float = TOL_VUF_PERCENT) -> pd.DataFrame:
    seq = compute_symmetrical_components(net)
    max_abs_error = seq["VUF_error_percent"].abs().max()
    print(f"Max VUF error = {max_abs_error:.3e} percent")
    if max_abs_error > atol:
        raise AssertionError(f"VUF cross-check failed: max error {max_abs_error} > {atol}")
    return seq


vuf_check = validate_vuf_against_pandapower(net)
display(vuf_check)

## 7. Proof 2 — Per-phase P/Q power balance

对每一相分别检查：

$$
P_{grid,\phi}+P_{sgen,\phi}-P_{load,\phi}-P_{loss,\phi}\approx 0
$$

$$
Q_{grid,\phi}+Q_{sgen,\phi}-Q_{load,\phi}-Q_{loss,\phi}\approx 0
$$

本周 feeder 没有 transformer，所以 losses 只来自 line。Week 3/4 若引入 transformer，需要把 transformer loss 也加入平衡式。

In [ ]:
def _safe_sum(df: pd.DataFrame, column: str) -> float:
    if df is None or len(df) == 0 or column not in df.columns:
        return 0.0
    return float(df[column].sum())


def validate_per_phase_power_balance(
    net: pp.pandapowerNet,
    atol_mw: float = TOL_POWER_MW,
    atol_mvar: float = TOL_POWER_MVAR,
) -> pd.DataFrame:
    """Validate per-phase active/reactive power balance for this line-only teaching feeder."""
    rows = []
    for ph in PHASES:
        p_grid = _safe_sum(net.res_ext_grid_3ph, f"p_{ph}_mw")
        q_grid = _safe_sum(net.res_ext_grid_3ph, f"q_{ph}_mvar")

        p_sgen = _safe_sum(net.res_asymmetric_sgen_3ph, f"p_{ph}_mw")
        q_sgen = _safe_sum(net.res_asymmetric_sgen_3ph, f"q_{ph}_mvar")

        p_load = _safe_sum(net.res_asymmetric_load_3ph, f"p_{ph}_mw")
        q_load = _safe_sum(net.res_asymmetric_load_3ph, f"q_{ph}_mvar")

        p_line_loss = _safe_sum(net.res_line_3ph, f"pl_{ph}_mw")
        q_line_loss = _safe_sum(net.res_line_3ph, f"ql_{ph}_mvar")

        p_residual = p_grid + p_sgen - p_load - p_line_loss
        q_residual = q_grid + q_sgen - q_load - q_line_loss

        rows.append(
            {
                "phase": ph,
                "p_grid_mw": p_grid,
                "p_sgen_mw": p_sgen,
                "p_load_mw": p_load,
                "p_line_loss_mw": p_line_loss,
                "p_residual_mw": p_residual,
                "q_grid_mvar": q_grid,
                "q_sgen_mvar": q_sgen,
                "q_load_mvar": q_load,
                "q_line_loss_mvar": q_line_loss,
                "q_residual_mvar": q_residual,
            }
        )

    balance = pd.DataFrame(rows).set_index("phase")
    max_p_res = balance["p_residual_mw"].abs().max()
    max_q_res = balance["q_residual_mvar"].abs().max()
    print(f"Max P residual = {max_p_res:.3e} MW")
    print(f"Max Q residual = {max_q_res:.3e} Mvar")

    if max_p_res > atol_mw:
        raise AssertionError(f"Per-phase P balance failed: {max_p_res} > {atol_mw}")
    if max_q_res > atol_mvar:
        raise AssertionError(f"Per-phase Q balance failed: {max_q_res} > {atol_mvar}")
    return balance


power_balance_check = validate_per_phase_power_balance(net)
display(power_balance_check)

## 8. Proof 3 — 用 $S=VI^*$ 复核线路相电流

每相复功率：

$$
S_{\phi}=P_{\phi}+jQ_{\phi}=V_{\phi} I_{\phi}^{*}
$$

取幅值：

$$
|I_{\phi}|=\frac{|S_{\phi}|}{|V_{\phi}|}
$$

单位换算：

$$
\frac{\mathrm{MVA}}{\mathrm{kV}} = \mathrm{kA}
$$

因为 `vn_kv` 是 line-to-line base，所以相电压基值为：

$$
V_{LN,base}=\frac{V_{LL,base}}{\sqrt{3}}
$$

In [ ]:
def validate_phase_currents_from_s_equals_vi(
    net: pp.pandapowerNet,
    atol_ka: float = TOL_CURRENT_KA,
) -> pd.DataFrame:
    """Recompute line phase currents from per-phase S and bus phase voltage."""
    rows = []
    for line_idx, line in net.line.iterrows():
        for side in ["from", "to"]:
            bus_col = "from_bus" if side == "from" else "to_bus"
            bus = int(line[bus_col])
            vn_ll_kv = float(net.bus.loc[bus, "vn_kv"])
            for ph in PHASES:
                p_mw = float(net.res_line_3ph.loc[line_idx, f"p_{ph}_{side}_mw"])
                q_mvar = float(net.res_line_3ph.loc[line_idx, f"q_{ph}_{side}_mvar"])
                s_mva = math.hypot(p_mw, q_mvar)
                v_ln_kv = float(net.res_bus_3ph.loc[bus, f"vm_{ph}_pu"]) * vn_ll_kv / math.sqrt(3)
                i_calc_ka = s_mva / v_ln_kv
                i_pp_ka = float(net.res_line_3ph.loc[line_idx, f"i_{ph}_{side}_ka"])
                rows.append(
                    {
                        "line": line_idx,
                        "side": side,
                        "phase": ph,
                        "bus": bus,
                        "s_mva": s_mva,
                        "v_ln_kv": v_ln_kv,
                        "i_calc_ka": i_calc_ka,
                        "i_pandapower_ka": i_pp_ka,
                        "error_ka": i_calc_ka - i_pp_ka,
                    }
                )

    check = pd.DataFrame(rows)
    max_error = check["error_ka"].abs().max()
    print(f"Max current error = {max_error:.3e} kA")
    if max_error > atol_ka:
        raise AssertionError(f"Line current cross-check failed: {max_error} > {atol_ka}")
    return check


current_check = validate_phase_currents_from_s_equals_vi(net)
display(current_check)

## 9. Proof 4 — 复核 line loading percent

对每条线路和每一相：

$$
loading_{\ell,\phi}=100\frac{I_{\ell,\phi}}{I^{max}_{\ell}}
$$

pandapower 的 `loading_percent` 是三相 loading 中的最大值：

$$
loading_{\ell}=\max_{\phi\in\{a,b,c\}} loading_{\ell,\phi}
$$

In [ ]:
def validate_line_loading_formula(
    net: pp.pandapowerNet,
    atol_percent: float = TOL_LOADING_PERCENT,
) -> pd.DataFrame:
    """Recompute per-phase and total line loading from line current limits."""
    rows = []
    for line_idx, line in net.line.iterrows():
        max_i_ka = float(line["max_i_ka"])
        phase_loadings_calc = []
        phase_loadings_pp = []
        for ph in PHASES:
            i_ka = float(net.res_line_3ph.loc[line_idx, f"i_{ph}_ka"])
            loading_calc = 100 * i_ka / max_i_ka
            loading_pp = float(net.res_line_3ph.loc[line_idx, f"loading_{ph}_percent"])
            phase_loadings_calc.append(loading_calc)
            phase_loadings_pp.append(loading_pp)
            rows.append(
                {
                    "line": line_idx,
                    "phase": ph,
                    "i_ka": i_ka,
                    "max_i_ka": max_i_ka,
                    "loading_calc_percent": loading_calc,
                    "loading_pandapower_percent": loading_pp,
                    "error_percent": loading_calc - loading_pp,
                }
            )

        total_calc = max(phase_loadings_calc)
        total_pp = float(net.res_line_3ph.loc[line_idx, "loading_percent"])
        rows.append(
            {
                "line": line_idx,
                "phase": "max",
                "i_ka": np.nan,
                "max_i_ka": max_i_ka,
                "loading_calc_percent": total_calc,
                "loading_pandapower_percent": total_pp,
                "error_percent": total_calc - total_pp,
            }
        )

    check = pd.DataFrame(rows)
    max_error = check["error_percent"].abs().max()
    print(f"Max loading error = {max_error:.3e} percent")
    if max_error > atol_percent:
        raise AssertionError(f"Line loading cross-check failed: {max_error} > {atol_percent}")
    return check


loading_check = validate_line_loading_formula(net)
display(loading_check)

## 10. Three-phase security summary

本周先做 base-case security summary。Week 4 会对每个 N-1 contingency 计算同样的指标并生成 label。

In [ ]:
def summarize_three_phase_security(net: pp.pandapowerNet, scenario_name: str = "base") -> pd.Series:
    rb = net.res_bus_3ph
    rl = net.res_line_3ph
    rg = net.res_ext_grid_3ph

    vm_cols = [f"vm_{ph}_pu" for ph in PHASES]

    summary = {
        "scenario": scenario_name,
        "converged": bool(net.converged),
        "min_vm_a_pu": float(rb["vm_a_pu"].min()),
        "min_vm_b_pu": float(rb["vm_b_pu"].min()),
        "min_vm_c_pu": float(rb["vm_c_pu"].min()),
        "min_vm_all_pu": float(rb[vm_cols].min().min()),
        "max_vm_all_pu": float(rb[vm_cols].max().max()),
        "max_unbalance_percent": float(rb["unbalance_percent"].max()),
        "max_loading_a_percent": float(rl["loading_a_percent"].max()),
        "max_loading_b_percent": float(rl["loading_b_percent"].max()),
        "max_loading_c_percent": float(rl["loading_c_percent"].max()),
        "max_loading_percent": float(rl["loading_percent"].max()),
        "p_grid_total_mw": float(rg[[f"p_{ph}_mw" for ph in PHASES]].sum().sum()),
        "q_grid_total_mvar": float(rg[[f"q_{ph}_mvar" for ph in PHASES]].sum().sum()),
    }

    summary["voltage_violation"] = summary["min_vm_all_pu"] < V_MIN_PU or summary["max_vm_all_pu"] > V_MAX_PU
    summary["loading_violation"] = summary["max_loading_percent"] > LOADING_MAX_PERCENT
    summary["unbalance_violation"] = summary["max_unbalance_percent"] > VUF_MAX_PERCENT
    summary["any_static_violation"] = bool(
        summary["voltage_violation"] or summary["loading_violation"] or summary["unbalance_violation"]
    )
    return pd.Series(summary)


base_summary = summarize_three_phase_security(net, scenario_name="base")
display(base_summary.to_frame("value"))

## 11. Visualization helpers

这些图不追求美观，重点是帮助学生快速观察：

- 哪一相电压最低；
- 哪一相线路 loading 最高；
- 哪个 bus 的 VUF 最大。

In [ ]:
def plot_three_phase_voltage_profile(net: pp.pandapowerNet, title: str = "Three-phase voltage profile") -> None:
    rb = net.res_bus_3ph
    x = np.arange(len(rb))
    plt.figure(figsize=(8, 4))
    for ph in PHASES:
        plt.plot(x, rb[f"vm_{ph}_pu"].to_numpy(), marker="o", label=f"Phase {ph.upper()}")
    plt.axhline(V_MIN_PU, linestyle="--", linewidth=1, label="V min threshold")
    plt.axhline(V_MAX_PU, linestyle="--", linewidth=1, label="V max threshold")
    plt.xticks(x, net.bus["name"].astype(str).to_list(), rotation=20)
    plt.ylabel("Voltage magnitude [pu]")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_three_phase_line_loading(net: pp.pandapowerNet, title: str = "Three-phase line loading") -> None:
    rl = net.res_line_3ph
    x = np.arange(len(rl))
    width = 0.22
    plt.figure(figsize=(8, 4))
    for offset, ph in zip([-width, 0.0, width], PHASES):
        plt.bar(x + offset, rl[f"loading_{ph}_percent"].to_numpy(), width=width, label=f"Phase {ph.upper()}")
    plt.axhline(LOADING_MAX_PERCENT, linestyle="--", linewidth=1, label="Loading threshold")
    plt.xticks(x, net.line["name"].astype(str).to_list(), rotation=20)
    plt.ylabel("Loading [%]")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_bus_unbalance(net: pp.pandapowerNet, title: str = "Voltage unbalance factor") -> None:
    rb = net.res_bus_3ph
    x = np.arange(len(rb))
    plt.figure(figsize=(8, 4))
    plt.bar(x, rb["unbalance_percent"].to_numpy())
    plt.axhline(VUF_MAX_PERCENT, linestyle="--", linewidth=1, label="VUF threshold")
    plt.xticks(x, net.bus["name"].astype(str).to_list(), rotation=20)
    plt.ylabel("VUF [%]")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_three_phase_voltage_profile(net, "Base-case three-phase voltage profile")
plot_three_phase_line_loading(net, "Base-case three-phase line loading")
plot_bus_unbalance(net, "Base-case voltage unbalance")

## 12. Cross-validation — balanced case with `runpp()` vs `runpp_3ph()`

当三相完全对称时，三相潮流结果应该与等效 balanced power flow 结果接近。这个测试可以帮助我们确认：

- bus nominal voltage 的理解正确；
- 每相功率与三相总功率的换算正确；
- line current / loading 的换算正确；
- `runpp_3ph` 的基础设置没有明显错误。

In [ ]:
def create_equivalent_balanced_power_flow_net() -> pp.pandapowerNet:
    """Create a balanced runpp() network equivalent to create_three_phase_teaching_feeder(False)."""
    net = pp.create_empty_network(name="week2_equivalent_balanced_feeder", sn_mva=1.0)

    b_pcc = pp.create_bus(net, vn_kv=0.4, name="PCC_0p4kV")
    b_1 = pp.create_bus(net, vn_kv=0.4, name="LV_Bus_1")
    b_2 = pp.create_bus(net, vn_kv=0.4, name="LV_Bus_2")

    pp.create_ext_grid(net, bus=b_pcc, vm_pu=1.0, va_degree=0.0, name="PCC_slack")

    for from_bus, to_bus, length_km, name in [
        (b_pcc, b_1, 0.08, "Line_0_PCC_to_Bus1"),
        (b_1, b_2, 0.06, "Line_1_Bus1_to_Bus2"),
    ]:
        pp.create_line_from_parameters(
            net,
            from_bus=from_bus,
            to_bus=to_bus,
            length_km=length_km,
            r_ohm_per_km=0.642,
            x_ohm_per_km=0.083,
            c_nf_per_km=210.0,
            max_i_ka=0.20,
            name=name,
        )

    # Equivalent total three-phase powers.
    pp.create_load(net, bus=b_1, p_mw=3 * 0.008, q_mvar=3 * 0.0016, name="Bus1_balanced_total_load")
    pp.create_load(net, bus=b_2, p_mw=3 * 0.006, q_mvar=3 * 0.0012, name="Bus2_balanced_total_load")
    pp.create_sgen(net, bus=b_2, p_mw=3 * 0.002, q_mvar=0.0, name="Bus2_balanced_total_PV")
    return net


def compare_balanced_runpp_and_runpp3ph() -> pd.DataFrame:
    net3 = create_three_phase_teaching_feeder(unbalanced=False)
    net1 = create_equivalent_balanced_power_flow_net()

    run_three_phase_power_flow(net3)
    pp.runpp(net1, calculate_voltage_angles=True, algorithm="nr", tolerance_mva=1e-8)
    if not net1.converged:
        raise AssertionError("Balanced runpp() did not converge.")

    rows = []
    for bus_idx in net1.bus.index:
        vm_1ph = float(net1.res_bus.loc[bus_idx, "vm_pu"])
        vm_3ph_mean = float(net3.res_bus_3ph.loc[bus_idx, ["vm_a_pu", "vm_b_pu", "vm_c_pu"]].mean())
        max_phase_spread = float(
            net3.res_bus_3ph.loc[bus_idx, ["vm_a_pu", "vm_b_pu", "vm_c_pu"]].max()
            - net3.res_bus_3ph.loc[bus_idx, ["vm_a_pu", "vm_b_pu", "vm_c_pu"]].min()
        )
        rows.append(
            {
                "type": "bus_voltage",
                "index": bus_idx,
                "balanced_runpp": vm_1ph,
                "three_phase_mean_or_value": vm_3ph_mean,
                "abs_error": abs(vm_1ph - vm_3ph_mean),
                "three_phase_spread": max_phase_spread,
            }
        )

    for line_idx in net1.line.index:
        loading_1ph = float(net1.res_line.loc[line_idx, "loading_percent"])
        loading_3ph = float(net3.res_line_3ph.loc[line_idx, "loading_percent"])
        rows.append(
            {
                "type": "line_loading",
                "index": line_idx,
                "balanced_runpp": loading_1ph,
                "three_phase_mean_or_value": loading_3ph,
                "abs_error": abs(loading_1ph - loading_3ph),
                "three_phase_spread": np.nan,
            }
        )

    comparison = pd.DataFrame(rows)
    max_vm_error = comparison.loc[comparison["type"] == "bus_voltage", "abs_error"].max()
    max_loading_error = comparison.loc[comparison["type"] == "line_loading", "abs_error"].max()
    max_phase_spread = comparison.loc[comparison["type"] == "bus_voltage", "three_phase_spread"].max()

    print(f"Max bus voltage error between runpp and runpp_3ph = {max_vm_error:.3e} pu")
    print(f"Max line loading error between runpp and runpp_3ph = {max_loading_error:.3e} percent")
    print(f"Max A/B/C voltage spread in balanced three-phase case = {max_phase_spread:.3e} pu")

    if max_vm_error > TOL_BALANCED_VM_PU:
        raise AssertionError("Balanced runpp vs runpp_3ph voltage cross-check failed.")
    if max_loading_error > TOL_BALANCED_LOADING_PERCENT:
        raise AssertionError("Balanced runpp vs runpp_3ph loading cross-check failed.")
    if max_phase_spread > TOL_BALANCED_VM_PU:
        raise AssertionError("Balanced runpp_3ph case is not sufficiently phase-balanced.")
    return comparison


balanced_cross_check = compare_balanced_runpp_and_runpp3ph()
display(balanced_cross_check)

## 13. Scenario experiments for engineering intuition

本周不做大规模数据集，只做几个小场景来建立工程直觉。

- `base`: 基准不平衡场景；
- `high_load`: 三相负荷整体升高；
- `phase_b_heavy_load`: B 相负荷显著升高；
- `single_phase_pv_a`: A 相 PV 增大；
- `stress_unbalance`: 强相间不平衡，用于观察 voltage/loading/VUF 越限。

In [ ]:
SCENARIOS = {
    "base": {
        "load_scale": {"a": 1.0, "b": 1.0, "c": 1.0},
        "sgen_scale": {"a": 1.0, "b": 1.0, "c": 1.0},
    },
    "high_load": {
        "load_scale": {"a": 1.6, "b": 1.6, "c": 1.6},
        "sgen_scale": {"a": 1.0, "b": 1.0, "c": 1.0},
    },
    "phase_b_heavy_load": {
        "load_scale": {"a": 0.9, "b": 2.5, "c": 0.9},
        "sgen_scale": {"a": 1.0, "b": 1.0, "c": 1.0},
    },
    "single_phase_pv_a": {
        "load_scale": {"a": 1.0, "b": 1.0, "c": 1.0},
        "sgen_scale": {"a": 4.0, "b": 1.0, "c": 1.0},
    },
    "stress_unbalance": {
        "load_scale": {"a": 0.8, "b": 3.0, "c": 0.8},
        "sgen_scale": {"a": 4.5, "b": 1.0, "c": 0.5},
    },
}


def apply_scenario(net: pp.pandapowerNet, scenario_spec: Dict[str, Dict[str, float]]) -> pp.pandapowerNet:
    """Apply per-phase scaling to asymmetric load and sgen tables."""
    load_scale = scenario_spec["load_scale"]
    sgen_scale = scenario_spec["sgen_scale"]

    for ph in PHASES:
        net.asymmetric_load[f"p_{ph}_mw"] *= load_scale[ph]
        net.asymmetric_load[f"q_{ph}_mvar"] *= load_scale[ph]
        net.asymmetric_sgen[f"p_{ph}_mw"] *= sgen_scale[ph]
        net.asymmetric_sgen[f"q_{ph}_mvar"] *= sgen_scale[ph]
    return net


def run_scenario(scenario_name: str, scenario_spec: Dict[str, Dict[str, float]]) -> Tuple[pp.pandapowerNet, pd.Series]:
    scenario_net = create_three_phase_teaching_feeder(unbalanced=True)
    apply_scenario(scenario_net, scenario_spec)
    run_three_phase_power_flow(scenario_net)
    summary = summarize_three_phase_security(scenario_net, scenario_name=scenario_name)
    return scenario_net, summary


def run_all_scenarios() -> Tuple[Dict[str, pp.pandapowerNet], pd.DataFrame]:
    nets = {}
    summaries = []
    for scenario_name, scenario_spec in SCENARIOS.items():
        scenario_net, summary = run_scenario(scenario_name, scenario_spec)
        nets[scenario_name] = scenario_net
        summaries.append(summary)
    summary_df = pd.DataFrame(summaries).set_index("scenario")
    return nets, summary_df


scenario_nets, scenario_summary = run_all_scenarios()
display(scenario_summary)

## 14. Scenario sanity checks

这些不是数学定理，而是工程直觉检查：

1. `high_load` 的最小电压应低于 `base`；
2. `single_phase_pv_a` 的外部电网总有功进口应低于 `base`；
3. `stress_unbalance` 的最大 VUF 应高于 `base`；
4. `phase_b_heavy_load` 中 B 相最低电压应明显低于 base 的 B 相最低电压。

In [ ]:
def run_scenario_sanity_checks(summary_df: pd.DataFrame) -> None:
    base = summary_df.loc["base"]

    if not (summary_df.loc["high_load", "min_vm_all_pu"] < base["min_vm_all_pu"]):
        raise AssertionError("Expected high_load to reduce the minimum voltage.")

    if not (summary_df.loc["single_phase_pv_a", "p_grid_total_mw"] < base["p_grid_total_mw"]):
        raise AssertionError("Expected single_phase_pv_a to reduce grid active power import.")

    if not (summary_df.loc["stress_unbalance", "max_unbalance_percent"] > base["max_unbalance_percent"]):
        raise AssertionError("Expected stress_unbalance to increase voltage unbalance.")

    if not (summary_df.loc["phase_b_heavy_load", "min_vm_b_pu"] < base["min_vm_b_pu"]):
        raise AssertionError("Expected phase_b_heavy_load to reduce phase-B voltage.")

    print("Scenario sanity checks passed.")


run_scenario_sanity_checks(scenario_summary)

In [ ]:
# Visualize selected scenarios.
for name in ["base", "phase_b_heavy_load", "single_phase_pv_a", "stress_unbalance"]:
    plot_three_phase_voltage_profile(scenario_nets[name], title=f"Voltage profile — {name}")
    plot_bus_unbalance(scenario_nets[name], title=f"Voltage unbalance — {name}")

## 15. Regression-test bundle — 一键复核所有 proof

科研代码中建议保留这样的 regression-test bundle：当你改动网络、参数、场景或特征提取代码后，重新运行所有 proof。只要其中一个断言失败，就先修正数据生成逻辑，再训练 AI。

In [ ]:
def run_all_week2_validation_tests() -> pd.DataFrame:
    """Run proof and cross-validation tests for base and all scenario networks."""
    rows = []

    all_nets = {"base_initial": net, **scenario_nets}
    for name, n in all_nets.items():
        validate_input_network(n)
        validate_result_tables(n)
        seq = validate_vuf_against_pandapower(n)
        bal = validate_per_phase_power_balance(n)
        cur = validate_phase_currents_from_s_equals_vi(n)
        load = validate_line_loading_formula(n)
        rows.append(
            {
                "case": name,
                "max_vuf_error_percent": float(seq["VUF_error_percent"].abs().max()),
                "max_p_balance_residual_mw": float(bal["p_residual_mw"].abs().max()),
                "max_q_balance_residual_mvar": float(bal["q_residual_mvar"].abs().max()),
                "max_current_error_ka": float(cur["error_ka"].abs().max()),
                "max_loading_error_percent": float(load["error_percent"].abs().max()),
            }
        )

    # Also rerun the balanced runpp vs runpp_3ph cross-check.
    comparison = compare_balanced_runpp_and_runpp3ph()
    rows.append(
        {
            "case": "balanced_runpp_vs_runpp3ph",
            "max_vuf_error_percent": np.nan,
            "max_p_balance_residual_mw": np.nan,
            "max_q_balance_residual_mvar": np.nan,
            "max_current_error_ka": np.nan,
            "max_loading_error_percent": float(comparison["abs_error"].max()),
        }
    )

    run_scenario_sanity_checks(scenario_summary)
    test_summary = pd.DataFrame(rows).set_index("case")
    print("All Week 2 validation tests passed.")
    return test_summary


validation_test_summary = run_all_week2_validation_tests()
display(validation_test_summary)

## 16. Export Week 2 summary and figures

Week 5 的 ML baseline 和 Week 7 的 GNN 会用到类似的 features。这里先导出一个很小的 scenario-level summary。

In [ ]:
output_dir = Path("week2_outputs_complete")
output_dir.mkdir(exist_ok=True)

scenario_summary_path = output_dir / "week2_scenario_security_summary.csv"
scenario_summary.to_csv(scenario_summary_path)

validation_summary_path = output_dir / "week2_validation_test_summary.csv"
validation_test_summary.to_csv(validation_summary_path)

print(f"Saved: {scenario_summary_path}")
print(f"Saved: {validation_summary_path}")

## 17. Student exercises

### Exercise 1 — Create a healthier base case

修改 `create_three_phase_teaching_feeder()` 中的负荷或线路长度，使 base-case 的：

```text
min_vm_all_pu > 0.97
max_unbalance_percent < 1.0
```

然后重新运行所有 proof。

---

### Exercise 2 — Create a voltage violation

修改 `SCENARIOS["stress_unbalance"]`，使某一相电压低于 0.90 pu，同时潮流仍然收敛。

---

### Exercise 3 — Create a high PV case

修改 `single_phase_pv_a`，观察 A 相电压变化。回答：

> 单相 PV 增大时，为什么并不一定只影响 A 相？

---

### Exercise 4 — Prepare for Week 3

如果 feeder 从 3 个 bus 扩展到几百个 bus，你会如何组织：

```text
bus-level features
line-level features
scenario-level features
```

这些会变成 Week 5 和 Week 7 的 AI input。

## 18. 本周小结

本周你已经完成了三相安全预测最关键的一步：**可信的三相潮流计算**。

后续课程中，这些内容会这样被复用：

- Week 3：把运行点扩展为大量 microgrid scenarios；
- Week 4：对每个 scenario 执行 N-1 contingency，生成 phase-aware violation labels；
- Week 5：把 base-case features 输入传统 ML；
- Week 6：把 voltage/current/VUF penalty 加入 physics-informed loss；
- Week 7：把 bus/line/phase/channel 信息放入 GNN。

如果本周 proof 没有通过，不要进入 AI 训练。先修复物理建模和数据生成。